# LICA-AI Study — 04 — Effect Sizes (Morris' d for pre-post control-group designs)

*Notebook generated: 2026-10-17 | nbformat v4 | Python 3.10+*

**Study:** From Error Correction to Meaning Construction: Evaluating LICA-AI as a Real-Time Cognitive Scaffold for L2 Academic Writing
**Design:** two-group pre-test/post-test (Control vs Experimental), N = 120 participants,
240 rating records (2 raters x 120 participant-phase units), 4 rubric dimensions:
Ideas, Cohesion, Appropriateness, Language.

Run `pip install pandas numpy scipy statsmodels pingouin matplotlib seaborn openpyxl` first.

md

In [ ]:
import numpy as np, pandas as pdwide = pd.read_csv('lica_ai_wide.csv')DIMS = ['Ideas','Cohesion','Appropriateness','Language','Overall']def morris_d(pre_e, post_e, pre_c, post_c):    nE, nC = len(pre_e), len(pre_c)    sp = np.sqrt(((nE-1)*pre_e.var(ddof=1) + (nC-1)*pre_c.var(ddof=1)) / (nE+nC-2))    gain_diff = (post_e.mean() - pre_e.mean()) - (post_c.mean() - pre_c.mean())    d = gain_diff / sp    J = 1 - 3/(4*(nE+nC-2) - 1)          # exact small-sample correction    se = np.sqrt((nE+nC)/(nE*nC) + d**2/(2*(nE+nC-2)))    return d*J, J, se, gain_diff, sprows = []for dim in DIMS:    pe = wide[(wide.group=='Experimental') & (wide.phase=='Pre')][dim]    qe = wide[(wide.group=='Experimental') & (wide.phase=='Post')][dim]    pc = wide[(wide.group=='Control') & (wide.phase=='Pre')][dim]    qc = wide[(wide.group=='Control') & (wide.phase=='Post')][dim]    d, J, se, gd, sp = morris_d(pe, qe, pc, qc)    rows.append(dict(dimension=dim, morris_d=round(d,3), J=round(J,4), SE=round(se,3),                     CI_low=round(d-1.96*se,3), CI_high=round(d+1.96*se,3),                     raw_gain_diff=round(gd,2), pooled_pre_sd=round(sp,2)))effect_table = pd.DataFrame(rows); effect_table

md

In [ ]:
lmm = pd.read_csv('table4_3_lmm_interaction.csv')merged = effect_table.merge(lmm[['dimension','interaction']], on='dimension')assert (np.sign(merged['morris_d']) == np.sign(merged['interaction'])).all(), 'direction mismatch'merged[['dimension','morris_d','interaction']]

md

In [ ]:
effect_table.to_csv('table4_4_morris_d.csv', index=False); effect_table